# Impacts of Timestep reduction

### Loading the data

In [17]:
import pickle
import numpy as np
import os

# ══════════════════════════════════════════════════════════════
# Create reduced time-span datasets from Hybrid-normalized data
# 6h, 9h, 12h, and 18h only
# No saving at this stage; keep datasets in memory
# ══════════════════════════════════════════════════════════════

SOURCE_DIR = "./final_split_data_HybridNorm"

# ---------------------------------------------------------
# Load split
# ---------------------------------------------------------
def load_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)

    return data["X"].astype(np.float32), data["y"]


# ---------------------------------------------------------
# Load full 24-hour Hybrid-normalized data
# ---------------------------------------------------------
X_train_full, y_train = load_split(os.path.join(SOURCE_DIR, "train_set.pkl"))
X_val_full,   y_val   = load_split(os.path.join(SOURCE_DIR, "val_set.pkl"))
X_test_full,  y_test  = load_split(os.path.join(SOURCE_DIR, "test_set.pkl"))

print("Loaded full 24-hour Hybrid-normalized data:")
print("Train:", X_train_full.shape, y_train.shape)
print("Val  :", X_val_full.shape,   y_val.shape)
print("Test :", X_test_full.shape,  y_test.shape)


# ---------------------------------------------------------
# Define reduced time-span variants
# Assumption: 5-minute cadence → 12 samples per hour
# ---------------------------------------------------------
TIMESTEPS_PER_HOUR = 12

timespan_configs = {
    "hybrid_6h":  6  * TIMESTEPS_PER_HOUR,   # 72 timesteps
    "hybrid_9h":  9  * TIMESTEPS_PER_HOUR,   # 108 timesteps
    "hybrid_12h": 12 * TIMESTEPS_PER_HOUR,   # 144 timesteps
    "hybrid_18h": 18 * TIMESTEPS_PER_HOUR,   # 216 timesteps
}


# ---------------------------------------------------------
# Create datasets in memory using the last T timesteps
# ---------------------------------------------------------
datasets_timespan = {}

for name, T in timespan_configs.items():

    if T > X_train_full.shape[1]:
        raise ValueError(
            f"{name} requires {T} timesteps, but the full sequence has only "
            f"{X_train_full.shape[1]} timesteps."
        )

    datasets_timespan[name] = {
        "X_train": X_train_full[:, -T:, :],
        "y_train": y_train,

        "X_val": X_val_full[:, -T:, :],
        "y_val": y_val,

        "X_test": X_test_full[:, -T:, :],
        "y_test": y_test,

        "timesteps": T,
        "hours": T // TIMESTEPS_PER_HOUR,
    }


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------
print(
    f"\n{'Dataset':<15} "
    f"{'Hours':>6} "
    f"{'Timesteps':>10} "
    f"{'Train shape':<25} "
    f"{'Val shape':<25} "
    f"{'Test shape':<25} "
    f"{'Class 0':>8} "
    f"{'Class 1':>8} "
    f"{'Ratio':>8}"
)

print("─" * 135)

for name, d in datasets_timespan.items():

    counts = np.bincount(d["y_train"].astype(int), minlength=2)

    if counts[1] == 0:
        ratio = np.inf
        ratio_str = "inf"
    else:
        ratio = counts[0] / counts[1]
        ratio_str = f"{ratio:.1f}x"

    print(
        f"{name:<15} "
        f"{d['hours']:>6} "
        f"{d['timesteps']:>10} "
        f"{str(d['X_train'].shape):<25} "
        f"{str(d['X_val'].shape):<25} "
        f"{str(d['X_test'].shape):<25} "
        f"{counts[0]:>8} "
        f"{counts[1]:>8} "
        f"{ratio_str:>8}"
    )


print("\n✅ Reduced time-span datasets are ready in memory.")
print("Use `datasets_timespan` for 6h, 9h, 12h, and 18h experiments.")
print("The full 24h Hybrid-normalized dataset is already available in `final_split_data_HybridNorm`.")

Loaded full 24-hour Hybrid-normalized data:
Train: (12455, 288, 10) (12455,)
Val  : (1780, 288, 10) (1780,)
Test : (3559, 288, 10) (3559,)

Dataset          Hours  Timesteps Train shape               Val shape                 Test shape                 Class 0  Class 1    Ratio
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
hybrid_6h            6         72 (12455, 72, 10)           (1780, 72, 10)            (3559, 72, 10)               12337      118   104.6x
hybrid_9h            9        108 (12455, 108, 10)          (1780, 108, 10)           (3559, 108, 10)              12337      118   104.6x
hybrid_12h          12        144 (12455, 144, 10)          (1780, 144, 10)           (3559, 144, 10)              12337      118   104.6x
hybrid_18h          18        216 (12455, 216, 10)          (1780, 216, 10)           (3559, 216, 10)              12337      118   104.6x

✅ Reduced time-span datasets

### Catch22 Vectorization for SVM

In [3]:
import numpy as np
from joblib import Parallel, delayed
from sktime.transformations.panel.catch22 import Catch22
from sklearn.impute import SimpleImputer
import warnings

# ══════════════════════════════════════════════════════════════
# CATCH22 FEATURE EXTRACTION — TIME-SPAN REDUCTION
# Uses datasets_timespan:
# hybrid_6h, hybrid_9h, hybrid_12h, hybrid_18h
# Keeps results only in memory as datasets_svm_timespan
# ══════════════════════════════════════════════════════════════

def _catch22_chunk(X_chunk: np.ndarray) -> np.ndarray:
    transformer = Catch22(catch24=False)
    return transformer.fit_transform(X_chunk).to_numpy()


def extract_catch22(
    X: np.ndarray,
    label: str = "",
    n_jobs: int = -1,
    chunk_size: int = 500
) -> np.ndarray:

    n_samples, n_timepoints, n_channels = X.shape

    if not np.isfinite(X).all():
        n_bad = (~np.isfinite(X)).sum()
        print(f"    ⚠️  {n_bad} non-finite values in {label} — replacing with per-channel median")

        X = X.copy()
        for c in range(n_channels):
            col = X[:, :, c]
            median_c = np.nanmedian(col)

            if not np.isfinite(median_c):
                median_c = 0.0

            col[~np.isfinite(col)] = median_c
            X[:, :, c] = col

    # sktime expects: (samples, channels, timesteps)
    X_sktime = X.transpose(0, 2, 1).astype(np.float64)

    chunks = [
        X_sktime[i:i + chunk_size]
        for i in range(0, n_samples, chunk_size)
    ]

    print(f"    {label} — {n_samples} samples in {len(chunks)} chunks …", flush=True)

    results = Parallel(n_jobs=n_jobs)(
        delayed(_catch22_chunk)(chunk)
        for chunk in chunks
    )

    X_feat = np.vstack(results)

    expected_cols = 22 * n_channels
    assert X_feat.shape == (n_samples, expected_cols), (
        f"Unexpected catch22 output shape: {X_feat.shape} "
        f"(expected ({n_samples}, {expected_cols}))"
    )

    return X_feat


# ── Sanity check ───────────────────────────────────────────────
print("Pre-extraction value range check:")
print(f"{'Dataset':<14} {'Hours':>6} {'Finite?':<10} {'Min':>15} {'Max':>15} {'NaN count':>12}")
print("─" * 85)

for dataset_name, d in datasets_timespan.items():
    X = d["X_train"]

    print(
        f"{dataset_name:<14} "
        f"{d['hours']:>6} "
        f"{str(np.isfinite(X).all()):<10} "
        f"{np.nanmin(X):>15.4f} "
        f"{np.nanmax(X):>15.4f} "
        f"{(~np.isfinite(X)).sum():>12}"
    )


# ── Extraction loop: no saving, memory only ────────────────────
print("\nExtracting Catch22 features for time-span datasets …")

datasets_svm_timespan = {}

for dataset_name, d in datasets_timespan.items():

    print(f"\n[{dataset_name}] extracting train …", flush=True)
    X_tr = extract_catch22(
        d["X_train"],
        label=f"{dataset_name}/train",
        n_jobs=-1,
        chunk_size=200
    )

    print(f"[{dataset_name}] extracting val …", flush=True)
    X_va = extract_catch22(
        d["X_val"],
        label=f"{dataset_name}/val",
        n_jobs=-1,
        chunk_size=200
    )

    print(f"[{dataset_name}] extracting test …", flush=True)
    X_te = extract_catch22(
        d["X_test"],
        label=f"{dataset_name}/test",
        n_jobs=-1,
        chunk_size=200
    )

    datasets_svm_timespan[dataset_name] = {
        "X_train": X_tr,
        "y_train": d["y_train"],

        "X_val": X_va,
        "y_val": d["y_val"],

        "X_test": X_te,
        "y_test": d["y_test"],

        "hours": d["hours"],
        "timesteps": d["timesteps"],
    }

    print(
        f"[{dataset_name}] ✓  "
        f"train {X_tr.shape}  "
        f"val {X_va.shape}  "
        f"test {X_te.shape}"
    )


print("\n✅ Catch22 extraction complete.")
print("    Shape : 22 features × 10 channels = 220 cols per sample")
print(f"    Time-span datasets : {list(datasets_svm_timespan.keys())}")


# ══════════════════════════════════════════════════════════════
# IMPUTE — CATCH22 FEATURES
# Fit imputer on train only, apply to val/test
# ══════════════════════════════════════════════════════════════

for dataset_name in list(datasets_svm_timespan.keys()):

    d = datasets_svm_timespan[dataset_name]

    X_tr, y_tr = d["X_train"].copy(), d["y_train"]
    X_va, y_va = d["X_val"].copy(),   d["y_val"]
    X_te, y_te = d["X_test"].copy(),  d["y_test"]

    n_bad_tr = (~np.isfinite(X_tr)).sum()
    n_bad_va = (~np.isfinite(X_va)).sum()
    n_bad_te = (~np.isfinite(X_te)).sum()

    if n_bad_tr > 0 or n_bad_va > 0 or n_bad_te > 0:

        print(
            f"  ⚠️  [{dataset_name}] non-finite values — "
            f"train: {n_bad_tr}  val: {n_bad_va}  test: {n_bad_te}  → imputing"
        )

        X_tr = np.where(np.isfinite(X_tr), X_tr, np.nan)
        X_va = np.where(np.isfinite(X_va), X_va, np.nan)
        X_te = np.where(np.isfinite(X_te), X_te, np.nan)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            imp = SimpleImputer(strategy="median")

            X_tr = imp.fit_transform(X_tr)
            X_va = imp.transform(X_va)
            X_te = imp.transform(X_te)

        X_tr = np.nan_to_num(X_tr, nan=0.0, posinf=0.0, neginf=0.0)
        X_va = np.nan_to_num(X_va, nan=0.0, posinf=0.0, neginf=0.0)
        X_te = np.nan_to_num(X_te, nan=0.0, posinf=0.0, neginf=0.0)

        print(f"  ✓  [{dataset_name}] imputed")

    else:
        print(f"  ✓  [{dataset_name}] no NaN or Inf — no imputation needed")

    datasets_svm_timespan[dataset_name] = {
        "X_train": X_tr.astype(np.float32),
        "y_train": y_tr,

        "X_val": X_va.astype(np.float32),
        "y_val": y_va,

        "X_test": X_te.astype(np.float32),
        "y_test": y_te,

        "hours": d["hours"],
        "timesteps": d["timesteps"],
    }


print(f"\n✅ datasets_svm_timespan ready: {list(datasets_svm_timespan.keys())}")

Pre-extraction value range check:
Dataset         Hours Finite?                Min             Max    NaN count
─────────────────────────────────────────────────────────────────────────────────────
hybrid_6h           6 True               -6.5136         27.2851            0
hybrid_9h           9 True               -6.5136         27.2851            0
hybrid_12h         12 True               -6.5136         27.2851            0
hybrid_18h         18 True               -6.5136         27.2851            0

Extracting Catch22 features for time-span datasets …

[hybrid_6h] extracting train …
    hybrid_6h/train — 12455 samples in 63 chunks …
[hybrid_6h] extracting val …
    hybrid_6h/val — 1780 samples in 9 chunks …
[hybrid_6h] extracting test …
    hybrid_6h/test — 3559 samples in 18 chunks …
[hybrid_6h] ✓  train (12455, 220)  val (1780, 220)  test (3559, 220)

[hybrid_9h] extracting train …
    hybrid_9h/train — 12455 samples in 63 chunks …
[hybrid_9h] extracting val …
    hybrid_9h/val

## Classification

### Helper Functions

In [4]:
# ══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.svm import SVC
import time
import os

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)


def compute_metrics(y_true, y_pred):
    cm             = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()

    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / (TP + TN + FP + FN)

    tss  = recall - FP / (FP + TN) if (FP + TN) > 0 else 0.0

    expected = ((TP + FN) * (TP + FP) + (TN + FP) * (TN + FN)) / (TP + TN + FP + FN) ** 2
    hss1     = (accuracy - expected) / (1 - expected) if (1 - expected) > 0 else 0.0

    denom = ((TP + FN) * (FN + TN) + (TP + FP) * (FP + TN))
    hss2  = 2 * (TP * TN - FP * FN) / denom if denom > 0 else 0.0

    hits_random = (TP + FP) * (TP + FN) / (TP + TN + FP + FN)
    gss = (TP - hits_random) / (TP + FP + FN - hits_random) if (TP + FP + FN - hits_random) > 0 else 0.0

    return {
        'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN),
        'tss': tss, 'hss1': hss1, 'hss2': hss2, 'gss': gss,
        'recall': recall, 'f1': f1, 'accuracy': accuracy
    }


def train_and_evaluate(model, X_train, y_train, X_test, y_test):
    t0         = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    t0         = time.time()
    y_pred     = model.predict(X_test)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test, y_pred)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics


def save_results(metrics_list, filepath):
    with open(filepath, 'w') as f:
        for m in metrics_list:
            line = (f"{m['TP']},{m['TN']},{m['FP']},{m['FN']},"
                    f"{m['tss']:.6f},{m['hss1']:.6f},{m['hss2']:.6f},{m['gss']:.6f},"
                    f"{m['recall']:.6f},{m['f1']:.6f},{m['accuracy']:.6f},"
                    f"{m['train_time']:.4f},{m['infer_time']:.4f}")
            f.write(line + "\n")


def print_results(metrics_list, title):
    keys = ['tss', 'hss1', 'hss2', 'gss', 'recall', 'f1', 'accuracy', 'train_time', 'infer_time']
    print(f"\n{'─'*55}")
    print(f"  {title}")
    print(f"{'─'*55}")
    for i, m in enumerate(metrics_list):
        print(f"  Run {i+1}: TP={m['TP']}  TN={m['TN']}  FP={m['FP']}  FN={m['FN']}")
        print(f"         TSS={m['tss']:.4f}  HSS1={m['hss1']:.4f}  HSS2={m['hss2']:.4f}  GSS={m['gss']:.4f}")
        print(f"         Recall={m['recall']:.4f}  F1={m['f1']:.4f}  Acc={m['accuracy']:.4f}")
        print(f"         Train={m['train_time']:.2f}s  Infer={m['infer_time']:.4f}s")
        print()
    print(f"  ── Average of {len(metrics_list)} runs ──")
    for k in keys:
        avg = np.mean([m[k] for m in metrics_list])
        print(f"  {k:<12} : {avg:.4f}")
    print(f"{'─'*55}")

### SVM

In [8]:
import os
import time
import copy
import warnings
import numpy as np

from sklearn.svm import SVC
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# ══════════════════════════════════════════════════════════════
# SVM FINAL EXPERIMENTS — TIME-SPAN REDUCTION
# Uses fixed best hyperparameters selected previously
# Best model: RBF-SVC, C=0.1, gamma=scale, class_weight=balanced
# ══════════════════════════════════════════════════════════════

# ---------------------------------------------------------
# Fixed best SVM hyperparameters
# ---------------------------------------------------------
best_model_class = SVC

best_params = {
    "kernel": "rbf",
    "C": 0.1,
    "gamma": "scale",
    "class_weight": "balanced",
    "cache_size": 2000
}

print("═" * 70)
print("Final SVM experiments on Hybrid time-span variants")
print("Using fixed best hyperparameters selected from hybrid validation")
print("═" * 70)
print(f"Model  : {best_model_class.__name__}")
print(f"Params : {best_params}")


# ---------------------------------------------------------
# Run final experiments on time-span variants
# ---------------------------------------------------------
all_timespan_results = {}

for dataset_key, d in datasets_svm_timespan.items():

    hours = d.get("hours", "NA")
    timesteps = d.get("timesteps", d["X_train"].shape[1])

    print("\n" + "─" * 70)
    print(f"Dataset   : {dataset_key}")
    print(f"Hours     : {hours}")
    print(f"Timesteps : {timesteps}")
    print(f"Model     : {best_model_class.__name__}")
    print(f"Params    : {best_params}")
    print("─" * 70)

    metrics_list = []

    for run in range(2):

        model = best_model_class(**copy.deepcopy(best_params))

        metrics = train_and_evaluate(
            model,
            d["X_train"], d["y_train"],
            d["X_test"],  d["y_test"]
        )

        metrics_list.append(metrics)

        print(
            f"Run {run + 1}: "
            f"TSS={metrics['tss']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"Recall={metrics['recall']:.4f} | "
            f"Train={metrics['train_time']:.2f}s"
        )

    all_timespan_results[dataset_key] = metrics_list

    filepath = os.path.join(RESULTS_DIR, f"svm_{dataset_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"SVM | {dataset_key}")


# ---------------------------------------------------------
# Select best time span by average TSS
# ---------------------------------------------------------
best_dataset = None
best_avg_tss = -np.inf

print("\n" + "═" * 70)
print("SVM time-span summary")
print("═" * 70)

print(
    f"{'Dataset':<15} "
    f"{'Avg TSS':>10} "
    f"{'Avg F1':>10} "
    f"{'Avg Recall':>12}"
)
print("─" * 55)

for dataset_key, metrics_list in all_timespan_results.items():

    avg_tss = np.mean([m["tss"] for m in metrics_list])
    avg_f1 = np.mean([m["f1"] for m in metrics_list])
    avg_recall = np.mean([m["recall"] for m in metrics_list])

    print(
        f"{dataset_key:<15} "
        f"{avg_tss:>10.4f} "
        f"{avg_f1:>10.4f} "
        f"{avg_recall:>12.4f}"
    )

    if avg_tss > best_avg_tss:
        best_avg_tss = avg_tss
        best_dataset = dataset_key


print("\nBest SVM time-span dataset:")
print(f"  Dataset : {best_dataset}")
print(f"  Avg TSS : {best_avg_tss:.4f}")
print(f"  Model   : {best_model_class.__name__}")
print(f"  Params  : {best_params}")

print("\nAll SVM time-span experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")

══════════════════════════════════════════════════════════════════════
Final SVM experiments on Hybrid time-span variants
Using fixed best hyperparameters selected from hybrid validation
══════════════════════════════════════════════════════════════════════
Model  : SVC
Params : {'kernel': 'rbf', 'C': 0.1, 'gamma': 'scale', 'class_weight': 'balanced', 'cache_size': 2000}

──────────────────────────────────────────────────────────────────────
Dataset   : hybrid_6h
Hours     : 6
Timesteps : 72
Model     : SVC
Params    : {'kernel': 'rbf', 'C': 0.1, 'gamma': 'scale', 'class_weight': 'balanced', 'cache_size': 2000}
──────────────────────────────────────────────────────────────────────
Run 1: TSS=0.4440 | F1=0.1020 | Recall=0.5294 | Train=10.24s
Run 2: TSS=0.4440 | F1=0.1020 | Recall=0.5294 | Train=10.37s

───────────────────────────────────────────────────────
  SVM | hybrid_6h
───────────────────────────────────────────────────────
  Run 1: TP=18  TN=3224  FP=301  FN=16
         TSS=0.444

### GRU 

In [18]:
# ══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS — PyTorch
# ══════════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import numpy as np
import time
import os
import warnings
warnings.filterwarnings('ignore')

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.3):
        super(GRUModel, self).__init__()
        self.gru      = nn.GRU(input_size, hidden_size, num_layers=num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.bn       = nn.BatchNorm1d(hidden_size)
        self.dropout  = nn.Dropout(dropout)
        self.fc1      = nn.Linear(hidden_size, 32)
        self.relu     = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2      = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out    = out[:, -1, :]
        out    = self.bn(out)
        out    = self.dropout(out)
        out    = self.relu(self.fc1(out))
        out    = self.dropout2(out)
        return self.fc2(out).squeeze()


def train_and_evaluate_gru(params, X_train, y_train, X_val, y_val, X_test, y_test, class_ratio=13):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_va = torch.tensor(X_val,   dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val,   dtype=torch.float32).to(device)
    X_te = torch.tensor(X_test,  dtype=torch.float32).to(device)

    input_size = X_train.shape[2]
    model      = GRUModel(input_size,
                          hidden_size = params["units"],
                          num_layers  = params["layers"],
                          dropout     = params["dropout"]).to(device)

    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer  = torch.optim.Adam(model.parameters(), lr=params["lr"])

    batch_size     = params["batch_size"]
    n_samples      = X_tr.shape[0]
    best_val_loss  = float('inf')
    patience_count = 0
    best_state     = None

    t0 = time.time()
    for epoch in range(50):
        model.train()
        perm = torch.randperm(n_samples)
        for i in range(0, n_samples, batch_size):
            idx    = perm[i:i + batch_size]
            xb, yb = X_tr[idx], y_tr[idx]
            optimizer.zero_grad()
            loss   = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_va), y_va).item()

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= 5:
                break

    train_time = time.time() - t0

    model.load_state_dict(best_state)
    model.eval()
    t0 = time.time()
    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()
    y_pred     = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics, model


Using device: mps


### Run GRU Experiments

In [23]:
import os
import numpy as np
import copy

# ══════════════════════════════════════════════════════════════
# GRU FINAL EXPERIMENTS — TIME-SPAN REDUCTION
# Uses fixed best hyperparameters selected previously
# Best params: units=128, layers=1, dropout=0.3, lr=0.001, batch_size=32
# ══════════════════════════════════════════════════════════════

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# Fixed best GRU hyperparameters
# ---------------------------------------------------------
best_params = {
    "units": 128,
    "layers": 1,
    "dropout": 0.3,
    "lr": 0.001,
    "batch_size": 32
}

print(f"{'═' * 70}")
print("  Classifier : GRU (PyTorch)")
print("  Experiment : Hybrid time-span reduction")
print(f"  Best params: {best_params}")
print(f"{'═' * 70}")


# ---------------------------------------------------------
# Run final GRU experiments on time-span variants
# ---------------------------------------------------------
all_gru_timespan_results = {}

for dataset_key, d in datasets_timespan.items():

    hours = d.get("hours", "NA")
    timesteps = d.get("timesteps", d["X_train"].shape[1])

    cr = (
        (d["y_train"] == 0).sum() /
        (d["y_train"] == 1).sum()
    )

    print("\n" + "─" * 70)
    print(f"Dataset   : {dataset_key}")
    print(f"Hours     : {hours}")
    print(f"Timesteps : {timesteps}")
    print(f"Ratio     : {cr:.1f}:1")
    print(f"Params    : {best_params}")
    print("─" * 70)

    metrics_list = []

    for run in range(2):

        print(f"Run {run + 1} …", flush=True)

        metrics, _ = train_and_evaluate_gru(
            best_params,
            d["X_train"], d["y_train"],
            d["X_val"],   d["y_val"],
            d["X_test"],  d["y_test"],
            class_ratio=cr
        )

        metrics_list.append(metrics)

        print(
            f"Run {run + 1}: "
            f"TSS={metrics['tss']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"Recall={metrics['recall']:.4f} | "
            f"Train={metrics['train_time']:.1f}s"
        )

    all_gru_timespan_results[dataset_key] = metrics_list

    filepath = os.path.join(RESULTS_DIR, f"gru_{dataset_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"GRU | {dataset_key}")


# ---------------------------------------------------------
# Select best time span by average TSS
# ---------------------------------------------------------
best_dataset = None
best_avg_tss = -np.inf

print("\n" + "═" * 70)
print("GRU time-span summary")
print("═" * 70)

print(
    f"{'Dataset':<15} "
    f"{'Hours':>6} "
    f"{'Steps':>8} "
    f"{'Avg TSS':>10} "
    f"{'Avg F1':>10} "
    f"{'Avg Recall':>12}"
)
print("─" * 70)

for dataset_key, metrics_list in all_gru_timespan_results.items():

    d = datasets_timespan[dataset_key]

    avg_tss = np.mean([m["tss"] for m in metrics_list])
    avg_f1 = np.mean([m["f1"] for m in metrics_list])
    avg_recall = np.mean([m["recall"] for m in metrics_list])

    print(
        f"{dataset_key:<15} "
        f"{d.get('hours', 'NA'):>6} "
        f"{d.get('timesteps', d['X_train'].shape[1]):>8} "
        f"{avg_tss:>10.4f} "
        f"{avg_f1:>10.4f} "
        f"{avg_recall:>12.4f}"
    )

    if avg_tss > best_avg_tss:
        best_avg_tss = avg_tss
        best_dataset = dataset_key


print("\nBest GRU time-span dataset:")
print(f"  Dataset : {best_dataset}")
print(f"  Avg TSS : {best_avg_tss:.4f}")
print(f"  Params  : {best_params}")

print("\n✅ All GRU time-span experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print(f"Files: {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

══════════════════════════════════════════════════════════════════════
  Classifier : GRU (PyTorch)
  Experiment : Hybrid time-span reduction
  Best params: {'units': 128, 'layers': 1, 'dropout': 0.3, 'lr': 0.001, 'batch_size': 32}
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
Dataset   : hybrid_6h
Hours     : 6
Timesteps : 72
Ratio     : 104.6:1
Params    : {'units': 128, 'layers': 1, 'dropout': 0.3, 'lr': 0.001, 'batch_size': 32}
──────────────────────────────────────────────────────────────────────
Run 1 …
Run 1: TSS=0.6281 | F1=0.1144 | Recall=0.7353 | Train=67.5s
Run 2 …
Run 2: TSS=0.6170 | F1=0.1050 | Recall=0.7353 | Train=84.0s

───────────────────────────────────────────────────────
  GRU | hybrid_6h
───────────────────────────────────────────────────────
  Run 1: TP=25  TN=3147  FP=378  FN=9
         TSS=0.6281  HSS1=0.0985  HSS2=0.0985  GSS=0.0518
         Recall=0.7353  F1=0.1144

### PatchTST

In [12]:
import os
import time
import copy
import warnings
import numpy as np

import torch
import torch.nn as nn
from transformers import PatchTSTConfig, PatchTSTForClassification

warnings.filterwarnings("ignore")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


def patchtst_label(p):
    return (
        f"patch={p['patch_length']}  "
        f"stride={p['patch_stride']}  "
        f"d={p['d_model']}  "
        f"heads={p['num_attention_heads']}  "
        f"layers={p['num_hidden_layers']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )


def build_patchtst_model(params, seq_len, n_channels):
    config = PatchTSTConfig(
        num_input_channels=n_channels,
        context_length=seq_len,
        num_targets=2,

        patch_length=params["patch_length"],
        patch_stride=params["patch_stride"],

        d_model=params["d_model"],
        num_attention_heads=params["num_attention_heads"],
        num_hidden_layers=params["num_hidden_layers"],
        ffn_dim=params["ffn_dim"],

        attention_dropout=params["dropout"],
        positional_dropout=params["dropout"],
        ff_dropout=params["dropout"],
        head_dropout=params["dropout"],

        pooling_type="mean",
        use_cls_token=True,
        norm_type="layernorm",
        channel_attention=True,
        problem_type="single_label_classification",
        scaling=None,
    )

    return PatchTSTForClassification(config)


def train_and_evaluate_patchtst(
    params,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    class_ratio,
    max_epochs=30,
    patience=3
):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.long).to(device)

    X_va = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val, dtype=torch.long).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    seq_len = X_train.shape[1]
    n_channels = X_train.shape[2]

    model = build_patchtst_model(
        params,
        seq_len=seq_len,
        n_channels=n_channels
    ).to(device)

    class_weights = torch.tensor(
        [1.0, float(class_ratio)],
        dtype=torch.float32
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params["lr"],
        weight_decay=1e-4
    )

    batch_size = params["batch_size"]
    n_samples = X_tr.shape[0]

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0

    t0 = time.time()

    for epoch in range(max_epochs):
        model.train()

        perm = torch.randperm(n_samples, device=device)

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]

            xb = X_tr[idx]
            yb = y_tr[idx]

            optimizer.zero_grad()

            outputs = model(past_values=xb)
            logits = outputs.prediction_logits

            loss = criterion(logits, yb)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_outputs = model(past_values=X_va)
            val_logits = val_outputs.prediction_logits
            val_loss = criterion(val_logits, y_va).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                break

    train_time = time.time() - t0

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    t0 = time.time()

    with torch.no_grad():
        test_outputs = model(past_values=X_te)
        test_logits = test_outputs.prediction_logits
        y_pred = torch.argmax(test_logits, dim=1).cpu().numpy()

    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred.astype(int))
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    return metrics, model

Using device: mps


In [13]:
# ══════════════════════════════════════════════════════════════
# PATCHTST FINAL EXPERIMENTS — TIME-SPAN REDUCTION
# Uses fixed best hyperparameters selected previously
# Best params: patch=12, stride=12, d=32, heads=4, layers=1,
#              dropout=0.2, lr=0.001, batch_size=128
# ══════════════════════════════════════════════════════════════

import os
import copy
import numpy as np

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# Fixed best PatchTST hyperparameters
# ---------------------------------------------------------
best_params = {
    "patch_length": 12,
    "patch_stride": 12,
    "d_model": 32,
    "num_attention_heads": 4,
    "num_hidden_layers": 1,
    "ffn_dim": 128,
    "dropout": 0.2,
    "lr": 0.001,
    "batch_size": 128,
}


def patchtst_label(p):
    return (
        f"patch={p['patch_length']}  "
        f"stride={p['patch_stride']}  "
        f"d={p['d_model']}  "
        f"heads={p['num_attention_heads']}  "
        f"layers={p['num_hidden_layers']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )


print(f"{'═' * 70}")
print("  Classifier : Hugging Face PatchTST")
print("  Experiment : Hybrid time-span reduction")
print(f"  Best params: {patchtst_label(best_params)}")
print(f"{'═' * 70}")


# ---------------------------------------------------------
# Run final PatchTST experiments on time-span variants
# ---------------------------------------------------------
all_patchtst_timespan_results = {}

for dataset_key, d in datasets_timespan.items():

    hours = d.get("hours", "NA")
    timesteps = d.get("timesteps", d["X_train"].shape[1])

    cr = (
        (d["y_train"] == 0).sum() /
        (d["y_train"] == 1).sum()
    )

    print("\n" + "─" * 70)
    print(f"Dataset   : {dataset_key}")
    print(f"Hours     : {hours}")
    print(f"Timesteps : {timesteps}")
    print(f"Ratio     : {cr:.1f}:1")
    print(f"Params    : {patchtst_label(best_params)}")
    print("─" * 70)

    metrics_list = []

    for run in range(2):

        print(f"Run {run + 1} …", flush=True)

        metrics, _ = train_and_evaluate_patchtst(
            best_params,
            d["X_train"], d["y_train"],
            d["X_val"],   d["y_val"],
            d["X_test"],  d["y_test"],
            class_ratio=cr,
            max_epochs=30,
            patience=3
        )

        metrics_list.append(metrics)

        print(
            f"Run {run + 1}: "
            f"TSS={metrics['tss']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"Recall={metrics['recall']:.4f} | "
            f"Train={metrics['train_time']:.1f}s"
        )

    all_patchtst_timespan_results[dataset_key] = metrics_list

    filepath = os.path.join(RESULTS_DIR, f"patchtst_{dataset_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"PatchTST | {dataset_key}")


# ---------------------------------------------------------
# Select best time span by average TSS
# ---------------------------------------------------------
best_dataset = None
best_avg_tss = -np.inf

print("\n" + "═" * 70)
print("PatchTST time-span summary")
print("═" * 70)

print(
    f"{'Dataset':<15} "
    f"{'Hours':>6} "
    f"{'Steps':>8} "
    f"{'Avg TSS':>10} "
    f"{'Avg F1':>10} "
    f"{'Avg Recall':>12}"
)

print("─" * 70)

for dataset_key, metrics_list in all_patchtst_timespan_results.items():

    d = datasets_timespan[dataset_key]

    avg_tss = np.mean([m["tss"] for m in metrics_list])
    avg_f1 = np.mean([m["f1"] for m in metrics_list])
    avg_recall = np.mean([m["recall"] for m in metrics_list])

    print(
        f"{dataset_key:<15} "
        f"{d.get('hours', 'NA'):>6} "
        f"{d.get('timesteps', d['X_train'].shape[1]):>8} "
        f"{avg_tss:>10.4f} "
        f"{avg_f1:>10.4f} "
        f"{avg_recall:>12.4f}"
    )

    if avg_tss > best_avg_tss:
        best_avg_tss = avg_tss
        best_dataset = dataset_key


print("\nBest PatchTST time-span dataset:")
print(f"  Dataset : {best_dataset}")
print(f"  Avg TSS : {best_avg_tss:.4f}")
print(f"  Params  : {patchtst_label(best_params)}")

print("\n✅ All PatchTST time-span experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print(f"Files: {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

══════════════════════════════════════════════════════════════════════
  Classifier : Hugging Face PatchTST
  Experiment : Hybrid time-span reduction
  Best params: patch=12  stride=12  d=32  heads=4  layers=1  drop=0.2  lr=0.001  bs=128
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
Dataset   : hybrid_6h
Hours     : 6
Timesteps : 72
Ratio     : 104.6:1
Params    : patch=12  stride=12  d=32  heads=4  layers=1  drop=0.2  lr=0.001  bs=128
──────────────────────────────────────────────────────────────────────
Run 1 …
Run 1: TSS=0.6335 | F1=0.1196 | Recall=0.7353 | Train=11.9s
Run 2 …
Run 2: TSS=0.6184 | F1=0.1062 | Recall=0.7353 | Train=13.6s

───────────────────────────────────────────────────────
  PatchTST | hybrid_6h
───────────────────────────────────────────────────────
  Run 1: TP=25  TN=3166  FP=359  FN=9
         TSS=0.6335  HSS1=0.1039  HSS2=0.1039  GSS=0.0548
         Recall=0.7353  

### InceptionTime

In [14]:
import os
import time
import copy
import warnings
import numpy as np

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


# ══════════════════════════════════════════════════════════════
# INCEPTIONTIME MODEL
# ══════════════════════════════════════════════════════════════

class InceptionModule(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels=32,
        kernel_sizes=(9, 19, 39),
        bottleneck_channels=32
    ):
        super(InceptionModule, self).__init__()

        if in_channels > 1:
            self.bottleneck = nn.Conv1d(
                in_channels,
                bottleneck_channels,
                kernel_size=1,
                bias=False
            )
            conv_in_channels = bottleneck_channels
        else:
            self.bottleneck = nn.Identity()
            conv_in_channels = in_channels

        self.conv_list = nn.ModuleList([
            nn.Conv1d(
                conv_in_channels,
                out_channels,
                kernel_size=k,
                padding=k // 2,
                bias=False
            )
            for k in kernel_sizes
        ])

        self.maxpool_conv = nn.Sequential(
            nn.MaxPool1d(kernel_size=3, stride=1, padding=1),
            nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False)
        )

        total_out_channels = out_channels * (len(kernel_sizes) + 1)

        self.bn = nn.BatchNorm1d(total_out_channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        x_bottleneck = self.bottleneck(x)

        conv_outputs = [
            conv(x_bottleneck)
            for conv in self.conv_list
        ]

        pool_output = self.maxpool_conv(x)

        out = torch.cat(conv_outputs + [pool_output], dim=1)
        out = self.bn(out)
        out = self.relu(out)

        return out


class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels=32, depth=3, use_residual=True):
        super(InceptionBlock, self).__init__()

        self.use_residual = use_residual
        self.depth = depth

        modules = []
        current_channels = in_channels

        for _ in range(depth):
            module = InceptionModule(
                in_channels=current_channels,
                out_channels=out_channels
            )
            modules.append(module)

            current_channels = out_channels * 4

        self.inception_modules = nn.ModuleList(modules)

        if use_residual:
            self.residual = nn.Sequential(
                nn.Conv1d(in_channels, current_channels, kernel_size=1, bias=False),
                nn.BatchNorm1d(current_channels)
            )
            self.relu = nn.ReLU()

    def forward(self, x):
        residual = x

        out = x
        for module in self.inception_modules:
            out = module(out)

        if self.use_residual:
            residual = self.residual(residual)
            out = self.relu(out + residual)

        return out


class InceptionTimeClassifier(nn.Module):
    def __init__(self, input_channels, out_channels=32, depth=6, dropout=0.3):
        super(InceptionTimeClassifier, self).__init__()

        block_depth = max(1, depth // 2)

        self.block1 = InceptionBlock(
            in_channels=input_channels,
            out_channels=out_channels,
            depth=block_depth,
            use_residual=True
        )

        block1_channels = out_channels * 4

        self.block2 = InceptionBlock(
            in_channels=block1_channels,
            out_channels=out_channels,
            depth=block_depth,
            use_residual=True
        )

        final_channels = out_channels * 4

        self.gap = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(final_channels, 1)

    def forward(self, x):
        # Input shape: (batch, timesteps, features)
        # Conv1d expects: (batch, features, timesteps)
        x = x.permute(0, 2, 1)

        x = self.block1(x)
        x = self.block2(x)

        x = self.gap(x).squeeze(-1)
        x = self.dropout(x)

        return self.fc(x).squeeze()


# ══════════════════════════════════════════════════════════════
# TRAIN AND EVALUATE INCEPTIONTIME
# ══════════════════════════════════════════════════════════════

def train_and_evaluate_inception(
    params,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    class_ratio,
    max_epochs=30,
    patience=3
):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_va = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val, dtype=torch.float32).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    input_channels = X_train.shape[2]

    model = InceptionTimeClassifier(
        input_channels=input_channels,
        out_channels=params["out_channels"],
        depth=params["depth"],
        dropout=params["dropout"]
    ).to(device)

    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params["lr"],
        weight_decay=1e-4
    )

    batch_size = params["batch_size"]
    n_samples = X_tr.shape[0]

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0

    t0 = time.time()

    for epoch in range(max_epochs):
        model.train()

        perm = torch.randperm(n_samples, device=device)

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]

            xb = X_tr[idx]
            yb = y_tr[idx]

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_va)
            val_loss = criterion(val_logits, y_va).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1

            if patience_count >= patience:
                break

    train_time = time.time() - t0

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    t0 = time.time()

    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()

    y_pred = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred.astype(int))
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    return metrics, model

Using device: mps


In [15]:
# ══════════════════════════════════════════════════════════════
# INCEPTIONTIME FINAL EXPERIMENTS — TIME-SPAN REDUCTION
# Uses fixed best hyperparameters selected previously
# Best params: channels=16, depth=6, dropout=0.3, lr=0.001, batch_size=128
# ══════════════════════════════════════════════════════════════

import os
import copy
import numpy as np

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# Fixed best InceptionTime hyperparameters
# ---------------------------------------------------------
best_params = {
    "out_channels": 16,
    "depth": 6,
    "dropout": 0.3,
    "lr": 0.001,
    "batch_size": 128
}


def inception_label(p):
    return (
        f"channels={p['out_channels']}  "
        f"depth={p['depth']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )


print(f"{'═' * 70}")
print("  Classifier : InceptionTime")
print("  Experiment : Hybrid time-span reduction")
print(f"  Best params: {inception_label(best_params)}")
print(f"{'═' * 70}")


# ---------------------------------------------------------
# Run final InceptionTime experiments on time-span variants
# ---------------------------------------------------------
all_inception_timespan_results = {}

for dataset_key, d in datasets_timespan.items():

    hours = d.get("hours", "NA")
    timesteps = d.get("timesteps", d["X_train"].shape[1])

    cr = (
        (d["y_train"] == 0).sum() /
        (d["y_train"] == 1).sum()
    )

    print("\n" + "─" * 70)
    print(f"Dataset   : {dataset_key}")
    print(f"Hours     : {hours}")
    print(f"Timesteps : {timesteps}")
    print(f"Ratio     : {cr:.1f}:1")
    print(f"Params    : {inception_label(best_params)}")
    print("─" * 70)

    metrics_list = []

    for run in range(2):

        print(f"Run {run + 1} …", flush=True)

        metrics, _ = train_and_evaluate_inception(
            best_params,
            d["X_train"], d["y_train"],
            d["X_val"],   d["y_val"],
            d["X_test"],  d["y_test"],
            class_ratio=cr,
            max_epochs=30,
            patience=3
        )

        metrics_list.append(metrics)

        print(
            f"Run {run + 1}: "
            f"TSS={metrics['tss']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"Recall={metrics['recall']:.4f} | "
            f"Train={metrics['train_time']:.1f}s"
        )

    all_inception_timespan_results[dataset_key] = metrics_list

    filepath = os.path.join(RESULTS_DIR, f"inceptiontime_{dataset_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"InceptionTime | {dataset_key}")


# ---------------------------------------------------------
# Select best time span by average TSS
# ---------------------------------------------------------
best_dataset = None
best_avg_tss = -np.inf

print("\n" + "═" * 70)
print("InceptionTime time-span summary")
print("═" * 70)

print(
    f"{'Dataset':<15} "
    f"{'Hours':>6} "
    f"{'Steps':>8} "
    f"{'Avg TSS':>10} "
    f"{'Avg F1':>10} "
    f"{'Avg Recall':>12}"
)

print("─" * 70)

for dataset_key, metrics_list in all_inception_timespan_results.items():

    d = datasets_timespan[dataset_key]

    avg_tss = np.mean([m["tss"] for m in metrics_list])
    avg_f1 = np.mean([m["f1"] for m in metrics_list])
    avg_recall = np.mean([m["recall"] for m in metrics_list])

    print(
        f"{dataset_key:<15} "
        f"{d.get('hours', 'NA'):>6} "
        f"{d.get('timesteps', d['X_train'].shape[1]):>8} "
        f"{avg_tss:>10.4f} "
        f"{avg_f1:>10.4f} "
        f"{avg_recall:>12.4f}"
    )

    if avg_tss > best_avg_tss:
        best_avg_tss = avg_tss
        best_dataset = dataset_key


print("\nBest InceptionTime time-span dataset:")
print(f"  Dataset : {best_dataset}")
print(f"  Avg TSS : {best_avg_tss:.4f}")
print(f"  Params  : {inception_label(best_params)}")

print("\n✅ All InceptionTime time-span experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print(f"Files: {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

══════════════════════════════════════════════════════════════════════
  Classifier : InceptionTime
  Experiment : Hybrid time-span reduction
  Best params: channels=16  depth=6  drop=0.3  lr=0.001  bs=128
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
Dataset   : hybrid_6h
Hours     : 6
Timesteps : 72
Ratio     : 104.6:1
Params    : channels=16  depth=6  drop=0.3  lr=0.001  bs=128
──────────────────────────────────────────────────────────────────────
Run 1 …
Run 1: TSS=0.3144 | F1=0.1319 | Recall=0.3529 | Train=21.4s
Run 2 …
Run 2: TSS=0.4488 | F1=0.1071 | Recall=0.5294 | Train=18.4s

───────────────────────────────────────────────────────
  InceptionTime | hybrid_6h
───────────────────────────────────────────────────────
  Run 1: TP=12  TN=3389  FP=136  FN=22
         TSS=0.3144  HSS1=0.1182  HSS2=0.1182  GSS=0.0628
         Recall=0.3529  F1=0.1319  Acc=0.9556
         Train=21.40s  Infer

### Saving the 6H data

In [16]:
import os
import pickle

# ══════════════════════════════════════════════════════════════
# Save Hybrid-normalized 6h dataset
# ══════════════════════════════════════════════════════════════

SAVE_DIR = "./final_split_data_HybridNorm_6h"
os.makedirs(SAVE_DIR, exist_ok=True)

# Get 6h dataset from memory
d6 = datasets_timespan["hybrid_6h"]

# ---------------------------------------------------------
# Save train / val / test separately
# Same format as your original files: {"X": ..., "y": ...}
# ---------------------------------------------------------
splits = {
    "train_set.pkl": {
        "X": d6["X_train"],
        "y": d6["y_train"]
    },
    "val_set.pkl": {
        "X": d6["X_val"],
        "y": d6["y_val"]
    },
    "test_set.pkl": {
        "X": d6["X_test"],
        "y": d6["y_test"]
    }
}

for filename, data in splits.items():
    path = os.path.join(SAVE_DIR, filename)

    with open(path, "wb") as f:
        pickle.dump(data, f)

    print(f"Saved: {path}")
    print("X shape:", data["X"].shape)
    print("y shape:", data["y"].shape)
    print("-" * 50)

print("\n✅ Hybrid 6h dataset saved successfully.")
print(f"Saved directory: {SAVE_DIR}")

Saved: ./final_split_data_HybridNorm_6h/train_set.pkl
X shape: (12455, 72, 10)
y shape: (12455,)
--------------------------------------------------
Saved: ./final_split_data_HybridNorm_6h/val_set.pkl
X shape: (1780, 72, 10)
y shape: (1780,)
--------------------------------------------------
Saved: ./final_split_data_HybridNorm_6h/test_set.pkl
X shape: (3559, 72, 10)
y shape: (3559,)
--------------------------------------------------

✅ Hybrid 6h dataset saved successfully.
Saved directory: ./final_split_data_HybridNorm_6h
